# Dopamine PINN: inverse-problem tuning (k-bias diagnosis)

Short companion to `dopamine_PINN.ipynb`. The full run at $T = 50$ ms gave a PINN $k$ error of $\approx 30\%$, while the FD reference estimator on the **same data** reached $7.7\%$. At $0\%$ noise the PINN is accurate ($k$ error $1.5\%$), so the bias is noise-induced. The hypothesis is that the PDE residual is under-weighted relative to the data term, letting the network absorb noise.

This notebook re-trains **only the inverse PINN** (single seed 1234, canonical 2%-noise data) for a few settings of the residual weight `W_RES` and the collocation count `N_DOMAIN`, and compares against:

| Reference on the same data | $D$ error | $k$ error |
|---|---|---|
| Main run, `W_RES = 10`, `N_DOMAIN = 10k` (seed 1234) | 11.82% | 30.16% |
| FD reference estimator (noise floor) | 6.96% | 7.71% |

Results are printed after each configuration and saved to `figures/tuning.json` after each one, so partial results survive a disconnect. Runtime on a T4: roughly 15 min per 10k-point configuration and 40 min per 30k-point configuration (about 1.5-2 h total).

**Run:** GPU runtime, then *Runtime → Run all* (the install cell restarts the kernel once; then *Run all* again).

## 0. Install dependencies

Installs JAX + matching CUDA plugin + Flax NNX + Optax + jaxopt all together (so the `jax` and `jaxlib`/CUDA-plugin versions stay in lockstep — installing only Flax can otherwise upgrade `jax` past what the preinstalled CUDA plugin supports and trigger `PJRT_FFI_UserData_Add_Args size: expected 48, got 40`).

**After this cell finishes, the kernel will auto-restart.** Then click **Runtime → Run all** to continue from Cell 2 onwards. Do NOT re-run this install cell after the restart — that wastes 2 minutes.

In [ ]:
import os
from pathlib import Path

# /tmp persists across kernel restarts within the same Colab VM session.
# The sentinel name is versioned: bump it whenever the pins below change,
# so a VM that already ran an older install cell re-installs.
SENTINEL = Path('/tmp/jax_pinn_deps_ready_v2')

if not SENTINEL.exists():
    # Pinned, mutually compatible versions. Do NOT use a bare --upgrade:
    # the newest flax (0.12.9) imports jax.experimental.hijax.HiPrimitive,
    # which the newest jax (0.11.2) removed, so an unpinned upgrade fails
    # at `from flax import nnx` with an AttributeError.
    # Flax NNX + Optax for PINN training, jaxopt for L-BFGS, Blackjax for HMC.
    rc = os.system('pip -q install '
                   '"jax[cuda12]==0.10.2" '
                   '"flax==0.12.5" '
                   '"optax>=0.2.3" '
                   '"jaxopt>=0.8" '
                   '"blackjax>=1.2.0" '
                   'scipy matplotlib')
    if rc != 0:
        raise RuntimeError('pip install failed - see the output above.')

    SENTINEL.touch()
    print('Install complete. Restarting kernel to load the new JAX libraries...')
    print('After the kernel restart, click Runtime -> Run all again.')
    print('This cell will be a no-op on the second run.')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed in this VM session - skipping.')

In [ ]:
import os, time, json, re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc                    # for Latin Hypercube Sampling

import jax
import jax.numpy as jnp
from flax import nnx
import optax
from jaxopt import LBFGS
from scipy.interpolate import RegularGridInterpolator

# float64 is required for k-parameter recovery in the inverse problem.
jax.config.update('jax_enable_x64', True)

SEED = 1234
np.random.seed(SEED)

FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

print(f'JAX version    : {jax.__version__}')
import flax
print(f'Flax version   : {flax.__version__}')
print(f'JAX devices    : {jax.devices()}')
print(f'Default backend: {jax.default_backend()}')
print(f'Default dtype  : float64 (required for inverse-problem accuracy)')
print(f'Sampling       : Latin Hypercube (interior collocation)')

## 1. Toggles and physical parameters

In [ ]:
# Toggles
QUICK        = False    # ~2-minute sanity check; not paper-quality
AUTO_FILL    = False    # upload B3.tex, substitute metrics, download
NOISE_SWEEP  = False    # additional inverse-problem sweep at 5 noise levels
                        # (~30-40 min extra on T4 -- set False to skip)

# Physical parameters (striatal dopamine; Cragg & Rice 2004, Trends Neurosci 27:270)
D_TRUE = 0.32     # effective diffusion coefficient (mu_m^2 / ms):
                  #   D* = D / lambda^2 = 0.763 / 1.54^2 = 0.322  (Cragg & Rice 2004, Box 1;
                  #   D* = D / lambda^2 relation: Nicholson & Phillips 1981)
K_TRUE = 0.020    # linearised DAT reuptake rate (1 / ms):
                  #   k' = Vmax / Km = (4.1 uM/s) / (0.21 uM) ~ 20 1/s = 0.020 1/ms
                  #   (Cragg & Rice 2004; the value used in their simulations)
L      = 5.0      # domain side length (mu_m); domain is [-L/2, L/2]^2.
                  #   Neighbouring DA synapse at r = 5 mu_m (Cragg & Rice 2004, Fig 2);
                  #   the zero-flux walls at +-L/2 are the symmetry planes between
                  #   two release sites 5 mu_m apart.
T      = 50.0     # simulation / observation window (ms): ~ 1/k = one DAT uptake
                  #   time constant at k' = 20 1/s (Cragg & Rice 2004). A 20 ms
                  #   window contains too little decay to identify k.
SIGMA  = 0.5      # release pulse width (mu_m) -- modelling choice
C0     = 1.0      # peak concentration scale (mu_M) -- normalisation; the PDE is
                  #   linear, so recovered (D, k) are independent of C0

# Deliberately offset initial guesses for the inverse problem
# (same relative offsets as before: D -6.25%, k -20%)
D_INIT = 0.30
K_INIT = 0.016

# Network and training hyperparameters
N_DOMAIN    = 10_000
N_BOUNDARY  = 400        # 100 per edge x 4 edges
N_INITIAL   = 400        # over the 2D square
N_TEST      = 5_000
LAYERS      = [3] + [64] * 4 + [1]   # input is (x, y, t)
ADAM_ITERS  = 20_000
LBFGS_ITERS = 2_000      # bumped from 500; needed to converge k tightly
LR          = 1e-3
W_RES       = 10.0       # inverse-problem PDE-residual loss weight (data, IC, BC weights = 1)
N_CHUNK     = 50

# Inverse-problem observation count.
N_OBS = 400

# Noise sweep levels (as percentage of peak C0).
# Skipped entirely if NOISE_SWEEP=False.
NOISE_LEVELS = [0.0, 1.0, 2.0, 5.0, 10.0]

if QUICK:
    N_DOMAIN, N_BOUNDARY, N_INITIAL, N_TEST = 1_000, 80, 80, 500
    ADAM_ITERS  = 1_000
    LBFGS_ITERS = 200
    N_CHUNK     = 25
    N_OBS = 80
    NOISE_LEVELS = [0.0, 2.0, 10.0]   # 3 levels for QUICK
    print('[QUICK MODE] Reduced hyperparameters - not paper-quality.')

assert ADAM_ITERS % N_CHUNK == 0, 'ADAM_ITERS must be a multiple of N_CHUNK'

print(f'D = {D_TRUE} mu_m^2/ms, k = {K_TRUE} 1/ms')
print(f'Domain: [-{L/2}, {L/2}]^2 mu_m, T = {T} ms')
print(f'Architecture: {LAYERS}, Adam iters: {ADAM_ITERS} ({ADAM_ITERS//N_CHUNK} chunks of {N_CHUNK})')
print(f'N_OBS = {N_OBS}, L-BFGS capped at {LBFGS_ITERS} iters')
print(f'Noise sweep: {"ON" if NOISE_SWEEP else "OFF"}, levels = {NOISE_LEVELS}')

## 3. Finite-difference reference solver (2D)

Explicit-time 5-point Laplacian stencil with Neumann (zero-flux) boundaries on all four edges. Stability condition in 2D: $\Delta t \le 1/(2D(\Delta x^{-2} + \Delta y^{-2}))$.

In [ ]:
def fd_reference(D=D_TRUE, k=K_TRUE, L=L, T=T, sigma=SIGMA, C0=C0,
                 nx=81, ny=81, nt=None, dt_store=0.02, verbose=True):
    """Explicit 2D FD solver with zero-flux (Neumann) boundaries.

    Returns (x, y, t_frames, frames). Only every `stride`-th time step is
    stored (frames ~dt_store ms apart), which bounds memory at T = 50 ms;
    observations are interpolated linearly in time between stored frames.
    """
    dx = L / (nx - 1)
    dy = L / (ny - 1)
    if nt is None:
        # 2D stability: dt <= 1 / (2D * (1/dx^2 + 1/dy^2)); 5% safety
        dt_max = 0.95 / (2.0 * D * (1.0/dx**2 + 1.0/dy**2))
        nt = int(np.ceil(T / dt_max)) + 1
    dt = T / (nt - 1)
    assert dt <= 1.0 / (2.0 * D * (1.0/dx**2 + 1.0/dy**2)), 'FD stability violated.'
    stride = max(1, int(round(dt_store / dt)))
    x = np.linspace(-L/2, L/2, nx)
    y = np.linspace(-L/2, L/2, ny)
    Xg, Yg = np.meshgrid(x, y, indexing='ij')
    C = C0 * np.exp(-(Xg**2 + Yg**2) / (2.0 * sigma**2))
    frames, t_frames = [C.copy()], [0.0]
    if verbose:
        print(f'FD: nx={nx}, ny={ny}, nt={nt}, dt={dt:.4e} ms, storing every {stride} steps')
    for n in range(1, nt):
        lap = np.zeros_like(C)
        # d^2/dx^2 with Neumann mirror at x = +-L/2
        lap[1:-1, :] += (C[2:, :] - 2*C[1:-1, :] + C[:-2, :]) / dx**2
        lap[0,   :]  += 2 * (C[1,  :] - C[0,  :]) / dx**2
        lap[-1,  :]  += 2 * (C[-2, :] - C[-1, :]) / dx**2
        # d^2/dy^2 with Neumann mirror at y = +-L/2
        lap[:, 1:-1] += (C[:, 2:] - 2*C[:, 1:-1] + C[:, :-2]) / dy**2
        lap[:, 0]    += 2 * (C[:, 1]  - C[:, 0])  / dy**2
        lap[:, -1]   += 2 * (C[:, -2] - C[:, -1]) / dy**2
        C = C + D * dt * lap - k * dt * C
        if n % stride == 0 or n == nt - 1:
            frames.append(C.copy())
            t_frames.append(n * dt)
    return x, y, np.array(t_frames), np.stack(frames)


# ---- FD reference ("oracle") estimator ------------------------------------
from scipy.optimize import least_squares

def fd_predict(D, k, x_o, y_o, t_o):
    """FD-solver concentrations at the observation points (x_o, y_o, t_o)."""
    x_g, y_g, t_g, H = fd_reference(D=D, k=k, verbose=False)
    interp = RegularGridInterpolator((t_g, x_g, y_g), H,
                                     bounds_error=False, fill_value=0.0)
    return interp(np.stack([t_o, x_o, y_o], axis=1))


def fd_fit(x_o, y_o, t_o, C_o, noise_pct=2.0, D0=None, k0=None):
    """Reference estimator: fit (D, k) with the FD solver itself.

    Nonlinear least squares on (log D, log k) using the same bounded-domain
    FD model that generated the synthetic data, so its error is set only by
    the observation noise: the best any method can do with these data.
    This is a deliberate 'inverse crime' and serves as a benchmark, not a
    competitor; the PINN never sees the FD solver.

    Returns D, k and the Gauss-Newton covariance of (log D, log k), i.e. the
    Laplace approximation of the posterior under a flat prior.
    """
    D0 = D_INIT if D0 is None else D0
    k0 = K_INIT if k0 is None else k0
    resid = lambda th: fd_predict(np.exp(th[0]), np.exp(th[1]), x_o, y_o, t_o) - C_o
    r = least_squares(resid, np.log([D0, k0]), diff_step=1e-3)
    # Noise variance: known for noisy data; residual-based for noise-free data
    s2 = (noise_pct / 100.0 * C0) ** 2 if noise_pct > 0 else float(np.mean(r.fun ** 2))
    cov_log = np.linalg.inv(r.jac.T @ r.jac) * s2
    return {'D': float(np.exp(r.x[0])), 'k': float(np.exp(r.x[1])),
            'cov_log': cov_log, 'nfev': int(r.nfev)}


t0 = time.time()
x_fd, y_fd, t_fd, C_fd_full = fd_reference()
print(f'FD run: {time.time()-t0:.1f} s, shape={C_fd_full.shape}, max C={C_fd_full.max():.4f} mu_M')

## 4. Forward PINN (2D, Flax NNX)

Network input is $(x, y, t) \in \mathbb{R}^3$, fed to the network unscaled (rescaling to $[-1, 1]^3$ was tested and degraded the fit of the fast early-time dynamics). Derivatives of $C$ with respect to each input are taken via `jax.grad`, giving the PDE residual at each collocation point:

$$\mathcal{L}_r(\theta) = \big| \partial_t \hat{C} - D\,(\partial_{xx} + \partial_{yy})\hat{C} + k\,\hat{C} \big|^2.$$

`nnx.jit` JIT-compiles the training step, and `nnx.value_and_grad` fuses forward + backward into one traced call. The Adam phase runs in Optax, and the second-order L-BFGS phase runs in jaxopt — both operating on the same NNX parameter pytree via `nnx.split`/`nnx.merge`.

In [ ]:
# ---- Flax NNX MLP ---------------------------------------------------------
class MLP(nnx.Module):
    """Fully-connected feedforward net with tanh activations.
    Input: (x, y, t) in R^3. Output: scalar C(x, y, t).

    All Linear layers carry param_dtype=float64 so the parameter pytree
    is dtype-homogeneous (matches the float64 D_log/k_log in InverseMLP
    and avoids 'Found more than one dtype in the tree' from jaxopt LBFGS).
    """
    def __init__(self, layers, *, rngs: nnx.Rngs):
        self.n_layers = len(layers) - 1
        for i in range(self.n_layers):
            setattr(self, f'lin_{i}',
                    nnx.Linear(layers[i], layers[i + 1],
                               kernel_init=nnx.initializers.glorot_normal(),
                               bias_init=nnx.initializers.zeros_init(),
                               param_dtype=jnp.float64,
                               rngs=rngs))

    def __call__(self, xyt):
        # Raw (x, y, t) inputs on purpose: rescaling them to [-1, 1]^3 was
        # tested and made the forward fit markedly worse (it compresses the
        # fast early-time dynamics, ~sigma^2 / 2D = 0.4 ms, into a steep
        # feature in scaled time).
        h = xyt
        for i in range(self.n_layers - 1):
            h = jnp.tanh(getattr(self, f'lin_{i}')(h))
        return getattr(self, f'lin_{self.n_layers - 1}')(h)


def C_at(model, x, y, t):
    return model(jnp.stack([x, y, t]))[0]


def pde_residual_one(model, x, y, t, D, k):
    """dC/dt - D Laplacian(C) + k C at one point."""
    dC_dt   = jax.grad(C_at, argnums=3)(model, x, y, t)
    d2C_dx2 = jax.grad(lambda u: jax.grad(C_at, argnums=1)(model, u, y, t))(x)
    d2C_dy2 = jax.grad(lambda v: jax.grad(C_at, argnums=2)(model, x, v, t))(y)
    return dC_dt - D * (d2C_dx2 + d2C_dy2) + k * C_at(model, x, y, t)


def bc_normal_deriv(model, x, y, t, nx, ny):
    dC_dx = jax.grad(C_at, argnums=1)(model, x, y, t)
    dC_dy = jax.grad(C_at, argnums=2)(model, x, y, t)
    return nx * dC_dx + ny * dC_dy


def sample_points(n_domain, n_initial, n_boundary, seed=SEED):
    """Collocation point sampling.

    Interior points (n_domain) are drawn by Latin Hypercube Sampling in
    the 3D (x, y, t) space. Initial and boundary points use uniform
    random sampling.
    """
    rng = np.random.default_rng(seed)

    # Latin Hypercube in 3D (x, y, t) -> [0,1]^3 -> physical domain
    lhs = qmc.LatinHypercube(d=3, seed=seed)
    u = lhs.random(n_domain)
    x_r = -L/2 + L * u[:, 0]
    y_r = -L/2 + L * u[:, 1]
    t_r =        T * u[:, 2]

    # IC points (t = 0): uniform 2D
    x_i = rng.uniform(-L/2, L/2, n_initial)
    y_i = rng.uniform(-L/2, L/2, n_initial)
    C_i = C0 * np.exp(-(x_i**2 + y_i**2) / (2*SIGMA**2))

    # Boundary: split evenly across the four edges (outward unit normals)
    per_edge = n_boundary // 4
    e_left  = (np.full(per_edge, -L/2), rng.uniform(-L/2, L/2, per_edge),
               rng.uniform(0, T, per_edge), np.full(per_edge, -1.0),
               np.full(per_edge,  0.0))
    e_right = (np.full(per_edge,  L/2), rng.uniform(-L/2, L/2, per_edge),
               rng.uniform(0, T, per_edge), np.full(per_edge,  1.0),
               np.full(per_edge,  0.0))
    e_bot   = (rng.uniform(-L/2, L/2, per_edge), np.full(per_edge, -L/2),
               rng.uniform(0, T, per_edge), np.full(per_edge, 0.0),
               np.full(per_edge, -1.0))
    e_top   = (rng.uniform(-L/2, L/2, per_edge), np.full(per_edge,  L/2),
               rng.uniform(0, T, per_edge), np.full(per_edge, 0.0),
               np.full(per_edge,  1.0))
    edges = [e_left, e_right, e_bot, e_top]
    x_b  = np.concatenate([e[0] for e in edges])
    y_b  = np.concatenate([e[1] for e in edges])
    t_b  = np.concatenate([e[2] for e in edges])
    nx_b = np.concatenate([e[3] for e in edges])
    ny_b = np.concatenate([e[4] for e in edges])

    return {k: jnp.asarray(v) for k, v in dict(
        x_r=x_r, y_r=y_r, t_r=t_r,
        x_i=x_i, y_i=y_i, C_i=C_i,
        x_b=x_b, y_b=y_b, t_b=t_b, nx_b=nx_b, ny_b=ny_b,
    ).items()}


def _component_losses(model, pts, D, k):
    res = jax.vmap(pde_residual_one, in_axes=(None, 0, 0, 0, None, None))(
        model, pts['x_r'], pts['y_r'], pts['t_r'], D, k)
    L_r = jnp.mean(res ** 2)

    C_pred_ic = jax.vmap(C_at, in_axes=(None, 0, 0, None))(
        model, pts['x_i'], pts['y_i'], jnp.float64(0.0))
    L_i = jnp.mean((C_pred_ic - pts['C_i']) ** 2)

    nd = jax.vmap(bc_normal_deriv, in_axes=(None, 0, 0, 0, 0, 0))(
        model, pts['x_b'], pts['y_b'], pts['t_b'], pts['nx_b'], pts['ny_b'])
    L_b = jnp.mean(nd ** 2)
    return L_r, L_i, L_b


def forward_loss(model, pts):
    L_r, L_i, L_b = _component_losses(model, pts, D_TRUE, K_TRUE)
    return 10.0 * L_r + L_i + L_b


pts = sample_points(N_DOMAIN, N_INITIAL, N_BOUNDARY)
print(f'Sampled {pts["x_r"].shape[0]} interior (LHS), '
      f'{pts["x_i"].shape[0]} IC, '
      f'{pts["x_b"].shape[0]} boundary points (dtype={pts["x_r"].dtype})')

## 5. Inverse PINN — recover $D$ and $k$ from noisy observations

400 observations sampled uniformly over $(x, y) \in \Omega$ and $t \in [0.5, 0.9T]$, with 2% Gaussian noise. Clean concentrations come from the FD reference (consistent with the PINN's Neumann BCs). Trainable parameters are log-parameterized to enforce positivity, and the residual loss is up-weighted $10\times$ vs. the data loss.

**Multi-seed evaluation.** To quantify run-to-run variability (rather than reporting a single anecdotal recovery), we re-train the inverse PINN over **5 different network-initialization seeds** with the *same* synthetic observation data. The final manuscript table reports mean ± std across seeds.

Runtime on Colab T4: ~60–90 min for 5 seeds (first seed pays the JIT compile cost; subsequent seeds run faster on the cached graph). Reduce `INVERSE_SEEDS` to 3 if you want to save time.

In [ ]:
def make_noisy_observations(n_obs=N_OBS, noise_pct=2.0, rng=None):
    """Sample noisy observations from the FD reference.

    Time window is [0.5, 0.9*T] = [0.5, 45] ms at T = 50 ms, i.e. about
    one uptake time constant (1/k = 50 ms at k = 0.020 1/ms). A 20 ms
    window contains too little decay to identify the reuptake rate k.
    """
    rng = rng or np.random.default_rng(SEED)
    obs_x = rng.uniform(-L/2, L/2, n_obs)
    obs_y = rng.uniform(-L/2, L/2, n_obs)
    obs_t = rng.uniform(0.5, 0.9 * T, n_obs)
    interp_fd = RegularGridInterpolator((t_fd, x_fd, y_fd), C_fd_full,
                                        bounds_error=False, fill_value=0.0)
    C_clean = interp_fd(np.stack([obs_t, obs_x, obs_y], axis=1))
    obs_C = C_clean + rng.normal(0.0, (noise_pct / 100.0) * C0, n_obs)
    return obs_x, obs_y, obs_t, obs_C

obs_x, obs_y, obs_t, obs_C = make_noisy_observations(n_obs=N_OBS, noise_pct=2.0)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(obs_x, obs_y, c=obs_t, cmap='plasma', s=20, alpha=0.7)
ax.set_xlabel('x (mu_m)'); ax.set_ylabel('y (mu_m)')
ax.set_xlim(-L/2, L/2); ax.set_ylim(-L/2, L/2)
ax.set_title(f'{len(obs_x)} noisy observations '
             f'(color = t in [0.5, {0.9*T:.0f}] ms, 2% noise)')
plt.colorbar(sc, ax=ax, label='t (ms)')
plt.show()

In [ ]:
# ---- Inverse model: MLP + log-parametrized D, k ---------------------------
class InverseMLP(nnx.Module):
    """MLP plus log-parametrized D and k as learnable scalars (float64)."""
    def __init__(self, layers, D0, k0, *, rngs: nnx.Rngs):
        self.mlp = MLP(layers, rngs=rngs)
        self.D_log = nnx.Param(jnp.asarray(np.log(D0), dtype=jnp.float64))
        self.k_log = nnx.Param(jnp.asarray(np.log(k0), dtype=jnp.float64))

    def __call__(self, xyt):
        return self.mlp(xyt)

    @property
    def D(self):
        getter = getattr(self.D_log, 'get_value', None)
        return jnp.exp(getter() if callable(getter) else self.D_log.value)

    @property
    def k(self):
        getter = getattr(self.k_log, 'get_value', None)
        return jnp.exp(getter() if callable(getter) else self.k_log.value)


def inverse_loss(model, pts, obs):
    D, k = model.D, model.k
    L_r, L_i, L_b = _component_losses(model.mlp, pts, D, k)
    C_pred_obs = jax.vmap(C_at, in_axes=(None, 0, 0, 0))(
        model.mlp, obs['x_d'], obs['y_d'], obs['t_d'])
    L_d = jnp.mean((C_pred_obs - obs['C_d']) ** 2)
    return W_RES * L_r + L_i + L_b + L_d


# ---- Build, train, and recover (per-seed) ---------------------------------
def train_inverse_once(obs, seed, write_variables_dat=False, return_model=False):
    """Run one full inverse-training pass with the given network-init seed.

    Returns (D_rec, k_rec, history) by default; if return_model=True, also
    returns the converged InverseMLP (needed by the Laplace UQ cell).
    """
    rngs = nnx.Rngs(seed)
    model = InverseMLP(LAYERS, D0=D_INIT, k0=K_INIT, rngs=rngs)
    optimizer = nnx.Optimizer(model, optax.adam(LR), wrt=nnx.Param)

    history = []

    @nnx.jit(static_argnames=('n_chunk',))
    def adam_chunk_inv(model, optimizer, pts, obs, n_chunk):
        last_loss = jnp.zeros(())
        for _ in range(n_chunk):
            loss, grads = nnx.value_and_grad(inverse_loss)(model, pts, obs)
            optimizer.update(model, grads)
            last_loss = loss
        return last_loss

    t0 = time.time()
    n_chunks = ADAM_ITERS // N_CHUNK
    for c in range(n_chunks):
        loss_val = adam_chunk_inv(model, optimizer, pts, obs, N_CHUNK)
        steps_done = (c + 1) * N_CHUNK
        if steps_done % 500 == 0:
            history.append((steps_done, float(model.D), float(model.k)))
        if c == 0 or (c + 1) % 40 == 0 or c == n_chunks - 1:
            elapsed = time.time() - t0
            print(f'    [seed {seed}] chunk {c+1:>4d}/{n_chunks}  '
                  f'step {steps_done:>5d}  loss = {float(loss_val):.4e}  '
                  f'D = {float(model.D):.4f}  k = {float(model.k):.4f}  '
                  f'elapsed = {elapsed:.1f}s')
    adam_time = time.time() - t0

    # ---- L-BFGS phase ----
    gdef_inv, state_inv = nnx.split(model)
    state_inv = jax.tree.map(
        lambda x: x.astype(jnp.float64) if hasattr(x, 'dtype') and jnp.issubdtype(x.dtype, jnp.floating) else x,
        state_inv,
    )

    def lbfgs_loss_inv(params):
        m = nnx.merge(gdef_inv, params)
        return inverse_loss(m, pts, obs)

    t0 = time.time()
    solver = LBFGS(fun=lbfgs_loss_inv, maxiter=LBFGS_ITERS, tol=1e-9)
    result = solver.run(state_inv)
    model = nnx.merge(gdef_inv, result.params)
    lbfgs_time = time.time() - t0

    D_rec, k_rec = float(model.D), float(model.k)
    history.append((ADAM_ITERS + LBFGS_ITERS, D_rec, k_rec))
    print(f'    [seed {seed}] Adam={adam_time:.1f}s, LBFGS={lbfgs_time:.1f}s, '
          f'D = {D_rec:.4f}, k = {k_rec:.4f}')

    if write_variables_dat:
        with open(FIG_DIR / 'variables.dat', 'w') as fh:
            for it, D, k in history:
                fh.write(f'{it}\t[{np.log(D):.6f}, {np.log(k):.6f}]\n')

    if return_model:
        return D_rec, k_rec, history, model
    return D_rec, k_rec, history

obs = {key: jnp.asarray(np.asarray(v, dtype=np.float64))
       for key, v in dict(x_d=obs_x, y_d=obs_y, t_d=obs_t, C_d=obs_C).items()}

## Tuning sweep

Each configuration re-trains the inverse PINN from scratch on the same observations. `W_RES` weights the PDE residual loss; data, IC and BC losses keep weight 1.

In [ ]:
# ---- Inverse-PINN tuning sweep: residual weight x collocation count ------
BASELINE = {'W_RES': 10.0, 'N_DOMAIN': 10_000, 'D_err': 11.82, 'k_err': 30.16}
FD_FLOOR = {'D_err': 6.96, 'k_err': 7.71}

TUNING_CONFIGS = [          # (W_RES, N_DOMAIN), most informative first
    (100.0,  10_000),
    (1000.0, 10_000),
    (10.0,   30_000),
    (100.0,  30_000),
]
if QUICK:
    TUNING_CONFIGS = [(100.0, 1_000), (10.0, 3_000)]

def _print_table(rows):
    print(f'\n  {"W_RES":>7} {"N_DOMAIN":>9} {"D":>8} {"|D err|%":>9} {"k":>9} {"|k err|%":>9} {"min":>6}')
    print(f'  {BASELINE["W_RES"]:>7.0f} {BASELINE["N_DOMAIN"]:>9d} {"":>8} {BASELINE["D_err"]:>8.2f}% '
          f'{"":>9} {BASELINE["k_err"]:>8.2f}%   (main run)')
    for r in rows:
        print(f'  {r["W_RES"]:>7.0f} {r["N_DOMAIN"]:>9d} {r["D_rec"]:>8.4f} {r["D_rel_err_pct"]:>8.2f}% '
              f'{r["k_rec"]:>9.5f} {r["k_rel_err_pct"]:>8.2f}% {r["minutes"]:>6.1f}')
    print(f'  {"FD reference (noise floor)":>35} {FD_FLOOR["D_err"]:>8.2f}% {"":>9} {FD_FLOOR["k_err"]:>8.2f}%')

tuning_results = []
for w_res, n_dom in TUNING_CONFIGS:
    print('\n' + '=' * 72)
    print(f'  W_RES = {w_res:g}, N_DOMAIN = {n_dom}')
    print('=' * 72)
    W_RES = w_res                                   # read by inverse_loss at trace time
    pts = sample_points(n_dom, N_INITIAL, N_BOUNDARY)   # read by train_inverse_once
    t0 = time.time()
    D_t, k_t, _ = train_inverse_once(obs, seed=SEED)
    tuning_results.append({
        'W_RES': w_res, 'N_DOMAIN': n_dom,
        'D_rec': D_t, 'k_rec': k_t,
        'D_rel_err_pct': 100.0 * abs(D_t - D_TRUE) / D_TRUE,
        'k_rel_err_pct': 100.0 * abs(k_t - K_TRUE) / K_TRUE,
        'minutes': (time.time() - t0) / 60.0,
    })
    with open(FIG_DIR / 'tuning.json', 'w') as fh:
        json.dump({'baseline_main_run': BASELINE, 'fd_reference': FD_FLOOR,
                   'seed': SEED, 'noise_pct': 2.0, 'n_obs': N_OBS, 'T_ms': T,
                   'results': tuning_results}, fh, indent=2)
    _print_table(tuning_results)

print(f'\nSaved {FIG_DIR / "tuning.json"}')
try:
    from google.colab import files as colab_files
    colab_files.download(str(FIG_DIR / 'tuning.json'))
except ImportError:
    print('(Not on Colab - tuning.json remains in figures/.)')